<a href="https://colab.research.google.com/github/mamathabaig/AI-lab/blob/main/1BM25CS476_Week9_FOL_Resolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import itertools
from collections import deque

# --- HELPER FUNCTIONS ---
def is_variable(term):
    return isinstance(term, str) and term[0].islower()

def parse_literal(lit_str):
    """Parses '~Pred(a, b)' into (is_negated, 'Pred', ['a', 'b'])"""
    lit_str = lit_str.strip()
    neg = lit_str.startswith("~")
    if neg:
        lit_str = lit_str[1:].strip()
    pred, args = lit_str.split("(")
    args = [a.strip() for a in args[:-1].split(",")]
    return neg, pred.strip(), args

def format_literal(neg, pred, args):
    prefix = "~" if neg else ""
    return f"{prefix}{pred}({', '.join(args)})"

def format_clause(clause):
    return " V ".join(clause) if clause else "{}"

# --- UNIFICATION ---
def unify(x, y, theta=None):
    if theta is None:
        theta = {}
    if theta is None:
        return None
    elif x == y:
        return theta
    elif is_variable(x):
        return unify_var(x, y, theta)
    elif is_variable(y):
        return unify_var(y, x, theta)
    elif isinstance(x, list) and isinstance(y, list) and len(x) == len(y):
        return unify(x[1:], y[1:], unify(x[0], y[0], theta))
    else:
        return None

def unify_var(var, x, theta):
    if var in theta:
        return unify(theta[var], x, theta)
    elif x in theta:
        return unify(var, theta[x], theta)
    else:
        new_theta = theta.copy()
        new_theta[var] = x
        return new_theta

def apply_subst(theta, clause):
    """Applies substitution dictionary theta to every literal in a clause."""
    result = []
    for lit in clause:
        neg, pred, args = parse_literal(lit)
        new_args = []
        for arg in args:
            while arg in theta:
                arg = theta[arg]
            new_args.append(arg)
        formatted = format_literal(neg, pred, new_args)
        if formatted not in result:  # Factoring duplicate literals
            result.append(formatted)
    return tuple(result)

# --- VARIABLE STANDARDIZATION ---
var_counter = itertools.count(1)
def standardize_clause(clause):
    mapping = {}
    idx = next(var_counter)
    new_clause = []
    for lit in clause:
        neg, pred, args = parse_literal(lit)
        new_args = []
        for arg in args:
            if is_variable(arg):
                if arg not in mapping:
                    mapping[arg] = f"{arg}_{idx}"
                new_args.append(mapping[arg])
            else:
                new_args.append(arg)
        new_clause.append(format_literal(neg, pred, new_args))
    return tuple(new_clause)

# --- RESOLUTION ENGINE ---
def resolve_pair(ci, cj):
    """Attempts to resolve two clauses and returns list of (resolvent, theta)."""
    ci = standardize_clause(ci)
    cj = standardize_clause(cj)
    resolvents = []

    for i, lit1 in enumerate(ci):
        neg1, pred1, args1 = parse_literal(lit1)
        for j, lit2 in enumerate(cj):
            neg2, pred2, args2 = parse_literal(lit2)

            # Complementary literals: same predicate, opposite signs
            if pred1 == pred2 and neg1 != neg2:
                theta = unify(args1, args2)
                if theta is not None:
                    rem_i = [ci[k] for k in range(len(ci)) if k != i]
                    rem_j = [cj[k] for k in range(len(cj)) if k != j]
                    resolvent = apply_subst(theta, rem_i + rem_j)
                    resolvents.append((resolvent, theta))
    return resolvents

def fol_resolution(kb_clauses, negated_goal):
    clauses = list(kb_clauses) + [negated_goal]
    seen = set(tuple(sorted(c)) for c in clauses)

    # Queue stores (clause, derivation_history)
    queue = deque([(negated_goal, [f"Negated Goal: {format_clause(negated_goal)}"])])

    while queue:
        current_clause, history = queue.popleft()

        for kb_clause in clauses:
            for resolvent, theta in resolve_pair(current_clause, kb_clause):
                step_desc = (
                    f"Resolve [{format_clause(current_clause)}] & "
                    f"[{format_clause(kb_clause)}] with {theta} "
                    f"=> {format_clause(resolvent)}"
                )
                new_history = history + [step_desc]

                # Empty clause found -> Contradiction!
                if len(resolvent) == 0:
                    return True, new_history

                canonical = tuple(sorted(resolvent))
                if canonical not in seen:
                    seen.add(canonical)
                    clauses.append(resolvent)
                    queue.append((resolvent, new_history))

    return False, []

# --- KNOWLEDGE BASE SETUP (CNF CLAUSES) ---
if __name__ == "__main__":
    # Constants start with Uppercase; variables start with lowercase; '~' denotes NOT
    kb_cnf = [
        ("~food(x)", "likes(John, x)"),                  # a. John likes all food
        ("food(Apple)",),                                # b1. Apple is food
        ("food(vegetables)",),                           # b2. Vegetables are food
        ("~eats(y, z)", "killed(y)", "food(z)"),         # c. Anything eaten & not killed is food
        ("eats(Anil, Peanuts)",),                        # d1. Anil eats peanuts
        ("alive(Anil)",),                                # d2. Anil is alive
        ("~eats(Anil, w)", "eats(Harry, w)"),            # e. Harry eats what Anil eats
        ("killed(g)", "alive(g)"),                       # f. Not killed -> alive
        ("~alive(k)", "~killed(k)")                      # g. Alive -> not killed
    ]

    negated_query = ("~likes(John, Peanuts)",)           # Negation of goal: likes(John, Peanuts)

    proved, proof_steps = fol_resolution(kb_cnf, negated_query)

    print("Proof Steps:")
    for idx, step in enumerate(proof_steps, 1):
        print(f"{idx}. {step}")
    print(f"\nConclusion 'likes(John, Peanuts)' Proved: {proved}")

Proof Steps:
1. Negated Goal: ~likes(John, Peanuts)
2. Resolve [~likes(John, Peanuts)] & [~food(x) V likes(John, x)] with {'x_2': 'Peanuts'} => ~food(Peanuts)
3. Resolve [~food(Peanuts)] & [food(Apple)] with {} => {}

Conclusion 'likes(John, Peanuts)' Proved: True
